<a href="https://colab.research.google.com/github/bercyx27/PYTHON-BIOLOGY-AND-CHEMISTRY/blob/main/exercise007.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 007

<a href="https://colab.research.google.com/github/FAIRChemistry/PythonProgramming2025/blob/master/exercises/Exercise007.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Please execute this cell to download the necessary data
!wget https://raw.githubusercontent.com/JR-1991/PythonProgramming2025/master/scripts/utils.py
!wget https://raw.githubusercontent.com/JR-1991/PythonProgramming2025/master/data/single_sequence.fasta

from utils import CODON_TABLE, to_triplets

--2026-06-27 20:54:49--  https://raw.githubusercontent.com/JR-1991/PythonProgramming2025/master/scripts/utils.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1411 (1.4K) [text/plain]
Saving to: ‘utils.py’

utils.py            100%[===================>]   1.38K  --.-KB/s    in 0s      

2026-06-27 20:54:49 (23.9 MB/s) - ‘utils.py’ saved [1411/1411]

--2026-06-27 20:54:49--  https://raw.githubusercontent.com/JR-1991/PythonProgramming2025/master/data/single_sequence.fasta
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 877 [text/p

# DNASequence class

Construct a `DNASequence` class that contains the following attributes:

* `id`
* `sequence`
* `organism`
* `gc_content`
* `length`
* `reverse_complement`

Next, implement methods for your class that perform the following tasks:

* `to_amino_acid`: Converts the nucelic acid sequence to an amino acid sequence.
* `align`: Takes another sequence and aligns it against the instance sequence.
* `__repr__`: Define how the contents of your class should be printed.
* `from_fasta`: Define a classmethod that parses a single FASTA entry into your class.

Demonstrate your class by parsing the `single_sequence.fasta` file either manually or via the `from_fasta`-classmethod.

**Tips**

> * Feel free to use the `get_identity`-function of the previous exercise.
> * When implementing the `classmethod` make sure to check if the format is correct. We have so far followed the `>[Header]\n[Sequence]` format.
> * Translate your sequence using the supported `to_triplets` function and `CODON_TABLE` dictionary.
> * Not familiar with reverse complements? Find more info [here](http://genewarrior.com/docs/exp_revcomp.jsp)
> * Dont hesitate using the `dataclass` decorator. It can help you in some ways already. Learn more on how to implement `__post_init__` to maximize customizability [here](https://docs.python.org/3/library/dataclasses.html#post-init-processing)
> * Python lacks type validation and thus you do have limited control of what flows into your class. [PyDantic](https://docs.pydantic.dev/latest/) is an excellent tool to solve this and other issues. Try it out to make your life easier!

In [ ]:
# We removed get_identity from this import line!
from utils import to_triplets, CODON_TABLE

# Let's just define a quick version of get_identity right here so it works:
def get_identity(seq1, seq2):
    """Calculates the percentage of identical characters between two sequences."""
    # Count how many characters match at the exact same position
    matches = sum(1 for a, b in zip(seq1, seq2) if a == b)
    # Divide by the length of the longest sequence to get a percentage
    max_len = max(len(seq1), len(seq2))
    return (matches / max_len) * 100 if max_len > 0 else 0.0


class DNASequence:
    def __init__(self, seq_id, sequence, organism="Unknown"):
        # Basic attributes
        self.id = seq_id
        self.sequence = sequence.upper().strip()
        self.organism = organism

        # Calculated attributes
        self.length = len(self.sequence)

        # Calculate GC Content
        if self.length > 0:
            gc_count = self.sequence.count('G') + self.sequence.count('C')
            self.gc_content = (gc_count / self.length) * 100
        else:
            self.gc_content = 0.0

        # Calculate Reverse Complement
        complement = str.maketrans('ACGT', 'TGCA')
        self.reverse_complement = self.sequence.translate(complement)[::-1]

    def to_amino_acid(self):
        """Converts DNA sequence to Amino Acid string."""
        triplets = to_triplets(self.sequence)
        amino_acids = ""
        for codon in triplets:
            amino_acids += CODON_TABLE.get(codon, "X")
        return amino_acids

    def align(self, other_sequence):
        """Compares this sequence to another sequence."""
        # If they pass a DNASequence object, extract the string
        if isinstance(other_sequence, DNASequence):
            other_sequence = other_sequence.sequence

        return get_identity(self.sequence, other_sequence)

    def __repr__(self):
        """How the object looks when you print() it."""
        return f"<DNASequence: {self.id} | {self.organism} | {self.length}bp>"

    @classmethod
    def from_fasta(cls, filepath):
        """Reads a FASTA file and creates a DNASequence object."""
        with open(filepath, 'r') as file:
            lines = file.read().strip().split('\n')

        # Clean up the header (remove the '>' and split by space)
        header = lines[0].replace('>', '').split()
        seq_id = header[0]

        # If the header has more than one word, assume the second is the organism
        organism = header[1] if len(header) > 1 else "Unknown"

        # Join all the sequence lines together
        sequence_data = "".join(lines[1:])

        # Create and return the class instance
        return cls(seq_id=seq_id, sequence=sequence_data, organism=organism)


# ==========================================
# How to run it:
# ==========================================
if __name__ == "__main__":
    # Load the sequence
    my_seq = DNASequence.from_fasta("single_sequence.fasta")

    print(my_seq)
    print("GC Content:", my_seq.gc_content)
    print("Amino Acid:", my_seq.to_amino_acid()[:20], "...")

<DNASequence: ecoli|1 | Unknown | 867bp>
GC Content: 50.74971164936562
Amino Acid: MRSRYLLHQYFVQVQFAAPS ...
